## Imporing and Mounting

In [20]:
# Cell 1 — Install
!pip install gradio transformers torch pillow requests -q

In [21]:
# Cell 2 — Mount Drive
from google.colab import drive
drive.mount("/content/drive")

import os, torch, requests
from io import BytesIO
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration

SAVE_DIR = "/content/drive/MyDrive/MuseumCaption"
DEMO_DIR = f"{SAVE_DIR}/demo_results"
os.makedirs(DEMO_DIR, exist_ok=True)
device   = "cuda" if torch.cuda.is_available() else "cpu"
print(f"GPU: {torch.cuda.get_device_name(0)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: NVIDIA A100-SXM4-40GB


## Loading Model - BLIP

In [22]:
# Cell 3 — Load BLIP
BLIP_PATH = f"{SAVE_DIR}/blip_model"

if os.path.exists(BLIP_PATH):
    print("Loading BLIP from Drive cache...")
    processor = BlipProcessor.from_pretrained(BLIP_PATH)
    model     = BlipForConditionalGeneration.from_pretrained(BLIP_PATH).to(device)
else:
    print("Downloading BLIP...")
    processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
    model     = BlipForConditionalGeneration.from_pretrained(
                    "Salesforce/blip-image-captioning-base").to(device)
    processor.save_pretrained(BLIP_PATH)
    model.save_pretrained(BLIP_PATH)

model.eval()
print("✓ BLIP ready.")

Loading BLIP from Drive cache...


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


✓ BLIP ready.


In [24]:
# Cell 4 — Core Functions
def run_caption(img, mode="single"):
    if img is None:
        return "⚠️ No image provided."
    inputs = processor(images=img, return_tensors="pt").to(device)
    with torch.no_grad():
        if mode == "single":
            out = model.generate(
                **inputs, max_new_tokens=50, min_new_tokens=8,
                num_beams=5, no_repeat_ngram_size=4,
                length_penalty=1.0
            )
        else:
            out = model.generate(
                **inputs, max_new_tokens=120, min_new_tokens=30,
                num_beams=5, no_repeat_ngram_size=4,
                length_penalty=2.0, repetition_penalty=2.0
            )
    caption = processor.decode(out[0], skip_special_tokens=True)
    with open(f"{DEMO_DIR}/demo_log.txt", "a") as f:
        f.write(f"{caption}\n")
    return caption

def run_vqa(img, question):
    if img is None:
        return "⚠️ Please provide an image."
    if not question or not question.strip():
        return "⚠️ Please enter a question."
    inputs = processor(images=img, text=question, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=30, min_new_tokens=2,
            num_beams=5, no_repeat_ngram_size=3,
        )
    answer = processor.decode(out[0], skip_special_tokens=True)
    q_lower = question.strip().lower().rstrip("?").strip()
    a_lower = answer.strip().lower()
    if a_lower.startswith(q_lower):
        answer = answer.strip()[len(question.strip()):].strip().lstrip("?.").strip()
    return answer if answer else "Could not determine an answer."

def caption_upload(img, mode):
    return run_caption(img, mode)

## UI- Gradio APP and Auto save to Drive

In [26]:
# %%
# Cell 5 — Gradio App
import gradio as gr
import textwrap, json
from datetime import datetime
import matplotlib.pyplot as plt

css = """
@import url('https://fonts.googleapis.com/css2?family=Plus+Jakarta+Sans:wght@300;400;600;800&display=swap');
* { font-family: 'Plus Jakarta Sans', sans-serif; }

.gradio-container {
    background-color: #030712 !important;
    max-width: 1100px !important;
    margin: 40px auto !important;
}
.header-box {
    background: linear-gradient(135deg, #0d1117 0%, #161b27 50%, #0d2137 100%);
    border-radius: 20px; padding: 36px 32px; text-align: center;
    margin-bottom: 24px; border: 1px solid #1e3a5f;
    box-shadow: 0 0 60px rgba(34,211,238,0.07);
}
.header-title {
    font-size: 2.8rem; font-weight: 900; letter-spacing: -1px;
    background: linear-gradient(90deg, #4ade80, #22d3ee, #818cf8);
    -webkit-background-clip: text; -webkit-text-fill-color: transparent;
}
.badge {
    display: inline-block;
    background: rgba(255,255,255,0.04); border: 1px solid #1e3a5f;
    border-radius: 20px; padding: 4px 14px;
    color: #94a3b8; font-size: 0.82rem; margin: 3px;
}
.github-btn {
    display: inline-block; margin-top: 14px;
    background: linear-gradient(135deg, #4ade80, #22d3ee);
    color: #0d1117 !important; font-weight: 800;
    font-size: 0.85rem; padding: 8px 20px;
    border-radius: 20px; text-decoration: none !important;
}
.custom-card {
    background: rgba(17, 24, 39, 0.7) !important;
    border: 1px solid rgba(255,255,255,0.08) !important;
    border-radius: 20px !important;
    padding: 20px; margin-bottom: 10px;
}
.action-btn {
    background: linear-gradient(135deg, #4ade80, #22d3ee) !important;
    color: #0d1117 !important; font-weight: 800 !important;
    border-radius: 12px !important; border: none !important;
    font-size: 1rem !important; padding: 14px !important;
    width: 100% !important; text-transform: uppercase !important;
    letter-spacing: 0.5px !important;
}
.section-label { color: #4ade80; font-weight: 700; font-size: 1rem; margin-bottom: 8px; }
textarea, input[type=text] {
    background: #0d1a2e !important; color: #e2e8f0 !important;
    border: 1px solid #1e3a5f !important; border-radius: 10px !important;
    font-size: 0.95rem !important;
}
.footer { text-align:center; color:#334155; font-size:0.8rem; padding:20px; }
"""

with gr.Blocks(css=css, theme=gr.themes.Soft(), title="VisionScript") as demo:

    # ── Header ────────────────────────────────────────────────
    gr.HTML("""
    <div class="header-box">
        <div class="header-title">🔭 VisionScript</div>
        <div style="color:#94a3b8; font-size:0.95rem; margin:4px 0 8px 0;">
            BLIP Image Captioning System
        </div>
        <div style="color:#64748b; font-size:0.88rem; line-height:1.6; margin:8px 0;">
            VisionScript is an AI-powered image captioning system that generates accurate captions
            for any image using BLIP — a vision-language model pretrained on 129M image-text pairs.
            Built comparing CLIP+GPT2, LoRA, Full Fine-tuning, and BLIP strategies.
        </div>
        <div style="margin:6px 0;">
            <span class="badge">👥 Group 5</span>
            <span class="badge">🎓 IE 7615 Deep Learning for AI</span>
            <span class="badge">🏫 Northeastern University</span>
        </div>
        <div style="margin:8px 0;">
            <a href="https://www.linkedin.com/in/simranasinha/" target="_blank"
               style="display:inline-block;margin:3px;background:linear-gradient(135deg,#4ade80,#22d3ee);
                      color:#0d1117!important;font-weight:700;font-size:0.82rem;
                      padding:6px 16px;border-radius:20px;text-decoration:none!important;">
                🔗 Simran Abhay Sinha</a>
            <a href="https://www.linkedin.com/in/sushmitha-sudharsan-2101/" target="_blank"
               style="display:inline-block;margin:3px;background:linear-gradient(135deg,#4ade80,#22d3ee);
                      color:#0d1117!important;font-weight:700;font-size:0.82rem;
                      padding:6px 16px;border-radius:20px;text-decoration:none!important;">
                🔗 Sushmitha Sudharsan</a>
            <a href="https://www.linkedin.com/in/aditya-kumar9512/" target="_blank"
               style="display:inline-block;margin:3px;background:linear-gradient(135deg,#4ade80,#22d3ee);
                      color:#0d1117!important;font-weight:700;font-size:0.82rem;
                      padding:6px 16px;border-radius:20px;text-decoration:none!important;">
                🔗 Aditya Kumar</a>
            <a href="https://www.linkedin.com/in/meghana-rao0109/" target="_blank"
               style="display:inline-block;margin:3px;background:linear-gradient(135deg,#4ade80,#22d3ee);
                      color:#0d1117!important;font-weight:700;font-size:0.82rem;
                      padding:6px 16px;border-radius:20px;text-decoration:none!important;">
                🔗 Meghana Rao</a>
        </div>
        <a class="github-btn"
           href="https://github.com/SimranaSinha/VisionScript-ImageCaptioning-BLIP"
           target="_blank">⭐ View on GitHub</a>
    </div>
    """)

    # ── Main Workspace ────────────────────────────────────────
    with gr.Row():

        # LEFT COLUMN
        with gr.Column(scale=1):
            with gr.Group(elem_classes="custom-card"):
                gr.HTML("<p class='section-label'>📸 Image Source</p>")
                input_img = gr.Image(
                    type="pil", label=None, show_label=False,
                    sources=["upload", "webcam", "clipboard"],
                    height=300
                )
                gr.HTML("<p class='section-label' style='margin-top:15px;'>⚙️ Caption Settings</p>")
                gr.HTML("""<span style='display:inline-block; background:#6d28d9; color:#e9d5ff;
                            border-radius:8px; padding:3px 12px; font-size:0.8rem;
                            font-weight:700; margin-bottom:6px;'>Detail Level</span>""")
                cap_mode = gr.Radio(
                    choices=["single", "paragraph"],
                    value="single", label="",
                    info="single = one sentence | paragraph = detailed description"
                )
                gen_cap_btn = gr.Button("✨ GENERATE CAPTION", elem_classes="action-btn")

        # RIGHT COLUMN
        with gr.Column(scale=1):

            # Caption Output
            with gr.Group(elem_classes="custom-card"):
                gr.HTML("<p class='section-label'>💡 AI Analysis</p>")
                gr.HTML("""<span style='display:inline-block; background:#6d28d9; color:#e9d5ff;
                            border-radius:8px; padding:3px 12px; font-size:0.8rem;
                            font-weight:700; margin-bottom:8px;'>Caption Output</span>""")
                out_caption = gr.Textbox(
                    label="", show_label=False, lines=6,
                    placeholder="Caption will appear here...",
                    interactive=False
                )

            # VQA Section
            with gr.Group(elem_classes="custom-card"):
                gr.HTML("<p class='section-label' style='color:#22d3ee;'>❓ Visual Q&A</p>")
                gr.HTML("""<span style='display:inline-block; background:#6d28d9; color:#e9d5ff;
                            border-radius:8px; padding:3px 12px; font-size:0.8rem;
                            font-weight:700; margin-bottom:8px;'>Ask a question about the image</span>""")
                vqa_msg = gr.Textbox(
                    label="", show_label=False,
                    placeholder="e.g. What color is the car? / How many people?",
                    lines=2
                )
                vqa_btn = gr.Button("🔍 Query Model", variant="secondary")
                gr.HTML("""<span style='display:inline-block; background:#6d28d9; color:#e9d5ff;
                            border-radius:8px; padding:3px 12px; font-size:0.8rem;
                            font-weight:700; margin:8px 0;'>Answer</span>""")
                out_vqa = gr.Textbox(
                    label="", show_label=False,
                    interactive=False, lines=2,
                    placeholder="Answer will appear here..."
                )

    # ── Footer ─────────────────────────────────────────────────
    gr.HTML("""
    <div class="footer">
        © 2026 VisionScript Team &nbsp;·&nbsp; Built with Gradio &amp; Salesforce BLIP &nbsp;·&nbsp;
        <a href="https://github.com/SimranaSinha/VisionScript-ImageCaptioning-BLIP"
           target="_blank" style="color:#4ade80; text-decoration:none;">GitHub ↗</a>
    </div>
    """)

    # ── Auto-save helper ───────────────────────────────────────
    def save_result(img, caption, mode="caption"):
        try:
            ts        = datetime.now().strftime("%Y%m%d_%H%M%S")
            save_name = f"VisionScript_{mode}_{ts}"
            fig, axes = plt.subplots(1, 2, figsize=(14, 6),
                                      gridspec_kw={"width_ratios": [1, 1.3]})
            axes[0].imshow(img)
            axes[0].set_title("Input Image", fontsize=12, fontweight="bold")
            axes[0].axis("off")
            axes[1].axis("off"); axes[1].set_xlim(0,1); axes[1].set_ylim(0,1)
            axes[1].add_patch(plt.Rectangle((0.03,0.08),0.94,0.84,
                              facecolor="#e6ffe6", edgecolor="#4ade80", linewidth=2))
            axes[1].text(0.5, 0.78, "Generated Caption", ha="center", va="center",
                         fontsize=12, fontweight="bold", color="#2e7d32")
            axes[1].text(0.5, 0.45, "\n".join(textwrap.wrap(caption, width=32)),
                         ha="center", va="center", fontsize=13,
                         linespacing=1.9, color="#1a1a1a")
            axes[1].text(0.5, 0.14, "Model: BLIP | VisionScript",
                         ha="center", va="center", fontsize=9,
                         color="#888", style="italic")
            plt.suptitle(f"VisionScript — {ts}", fontsize=13, fontweight="bold")
            plt.tight_layout()
            plt.savefig(f"{DEMO_DIR}/{save_name}.png", dpi=150, bbox_inches="tight")
            plt.close()
            with open(f"{DEMO_DIR}/{save_name}.json", "w") as jf:
                json.dump({"timestamp": ts, "caption": caption,
                            "mode": mode, "model": "BLIP"}, jf, indent=2)
            print(f"✓ Saved: {save_name}")
        except Exception as e:
            print(f"Save error: {e}")

    # ── Event Handlers ─────────────────────────────────────────
    def on_caption(img, mode):
        cap = caption_upload(img, mode)
        if img is not None and "⚠️" not in cap:
            save_result(img, cap, mode)
        return cap

    def on_vqa(img, question):
        ans = run_vqa(img, question)
        if img is not None and "⚠️" not in ans:
            save_result(img, f"Q: {question}\nA: {ans}", "vqa")
        return ans

    gen_cap_btn.click(fn=on_caption, inputs=[input_img, cap_mode], outputs=out_caption)
    vqa_btn.click(fn=on_vqa, inputs=[input_img, vqa_msg], outputs=out_vqa)

demo.launch(share=True, debug=False)

/tmp/ipykernel_552/4027162406.py:64: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=css, theme=gr.themes.Soft(), title="VisionScript") as demo:
/tmp/ipykernel_552/4027162406.py:64: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=css, theme=gr.themes.Soft(), title="VisionScript") as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f5acdfd336db95b20c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
